# HIT401 Simple Scam Email Project — Part 4

## Modern model comparison
Compare the traditional TF-IDF baseline with a pretrained MiniLM transformer.

**Important:** MiniLM is used as a frozen text-embedding model. We do not fine-tune its internal transformer weights. Logistic Regression learns from the resulting contextual embeddings. This is simpler and faster than full transformer fine-tuning, but it must not be described as a fine-tuned transformer.

## Before running
In Colab, select **Runtime → Change runtime type → T4 GPU** if available. The notebook will still run on CPU, but embedding thousands of emails will take longer.

In [ ]:
# Cell 1: Install Sentence Transformers
!pip -q install sentence-transformers

In [ ]:
# Cell 2: Import the tools
from pathlib import Path
import zipfile
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

## Step 1 — Upload the prepared data
Use the same `hit401_prepared_data.zip` created in Part 1. Using the same data and random split makes the model comparison fair.

In [ ]:
# Cell 3: Upload and extract the prepared data
zip_path = Path('/content/hit401_prepared_data.zip')
if not zip_path.exists():
    from google.colab import files
    print('Choose hit401_prepared_data.zip from your computer.')
    uploaded = files.upload()
    if 'hit401_prepared_data.zip' not in uploaded:
        raise FileNotFoundError('Please upload hit401_prepared_data.zip.')

data_dir = Path('/content/hit401_part4_data')
with zipfile.ZipFile(zip_path, 'r') as archive:
    archive.extractall(data_dir)

training_data = pd.read_csv(data_dir / 'training_data.csv')
external_test_data = pd.read_csv(data_dir / 'external_test_data.csv')
print('Training-source rows:', len(training_data))
print('External-test rows:', len(external_test_data))

In [ ]:
# Cell 4: Recreate the exact 80/20 internal split from Part 2
indices = np.arange(len(training_data))
develop_idx, internal_idx = train_test_split(
    indices, test_size=0.20, random_state=42, stratify=training_data['label']
)

develop = training_data.iloc[develop_idx].reset_index(drop=True)
internal = training_data.iloc[internal_idx].reset_index(drop=True)
external = external_test_data.reset_index(drop=True)

print('Model-training rows:', len(develop))
print('Internal-validation rows:', len(internal))
print('External-test rows:', len(external))

## Step 2 — Rebuild the traditional baseline
TF-IDF mainly represents which words and phrases appear. Logistic Regression learns weights for those features.

In [ ]:
# Cell 5: Train TF-IDF + Logistic Regression
tfidf = TfidfVectorizer(max_features=10000, stop_words='english', ngram_range=(1, 2), min_df=2)
X_dev_tfidf = tfidf.fit_transform(develop['text'])
X_internal_tfidf = tfidf.transform(internal['text'])
X_external_tfidf = tfidf.transform(external['text'])

baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_dev_tfidf, develop['label'])

baseline_internal_pred = baseline_model.predict(X_internal_tfidf)
baseline_external_pred = baseline_model.predict(X_external_tfidf)
baseline_external_prob = baseline_model.predict_proba(X_external_tfidf)[:, 1]
print('Baseline completed.')

## Step 3 — Create MiniLM transformer embeddings
`all-MiniLM-L6-v2` is a pretrained Sentence Transformers model. It maps each email to a 384-number vector intended to represent contextual meaning. Long emails are truncated to the model's input limit, which is an important limitation.

In [ ]:
# Cell 6: Download the pretrained MiniLM model
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
embedding_model = SentenceTransformer(MODEL_NAME, device=DEVICE)
print('Model:', MODEL_NAME)
print('Maximum input length:', embedding_model.max_seq_length, 'tokens')

In [ ]:
# Cell 7: Convert each email into a contextual embedding
start_time = time.time()
X_dev_mini = embedding_model.encode(
    develop['text'].tolist(), batch_size=32, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True
)
X_internal_mini = embedding_model.encode(
    internal['text'].tolist(), batch_size=32, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True
)
X_external_mini = embedding_model.encode(
    external['text'].tolist(), batch_size=32, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True
)

print('Embedding shape:', X_dev_mini.shape)
print('Embedding time in minutes:', round((time.time() - start_time) / 60, 2))

## Step 4 — Train the classifier on transformer embeddings
MiniLM supplies the contextual representations. Logistic Regression learns the scam-versus-legitimate boundary from our labelled training emails.

In [ ]:
# Cell 8: Train and predict
transformer_classifier = LogisticRegression(max_iter=1000, random_state=42)
transformer_classifier.fit(X_dev_mini, develop['label'])

transformer_internal_pred = transformer_classifier.predict(X_internal_mini)
transformer_external_pred = transformer_classifier.predict(X_external_mini)
transformer_external_prob = transformer_classifier.predict_proba(X_external_mini)[:, 1]
print('Transformer-embedding classifier completed.')

## Step 5 — Compare the models
The comparison uses identical labels and email groups. For scam detection, pay particular attention to external-test recall and F1-score.

In [ ]:
# Cell 9: Calculate comparison metrics
def metric_row(model_name, test_name, actual, predicted):
    return {
        'model': model_name, 'test': test_name,
        'accuracy': accuracy_score(actual, predicted),
        'precision_scam': precision_score(actual, predicted, zero_division=0),
        'recall_scam': recall_score(actual, predicted, zero_division=0),
        'f1_scam': f1_score(actual, predicted, zero_division=0),
    }

comparison = pd.DataFrame([
    metric_row('TF-IDF baseline', 'Internal validation', internal['label'], baseline_internal_pred),
    metric_row('MiniLM embeddings', 'Internal validation', internal['label'], transformer_internal_pred),
    metric_row('TF-IDF baseline', 'External test', external['label'], baseline_external_pred),
    metric_row('MiniLM embeddings', 'External test', external['label'], transformer_external_pred),
])
comparison.round(4)

In [ ]:
# Cell 10: Plot external-test metrics
external_comparison = comparison.loc[comparison['test'] == 'External test'].set_index('model')
external_comparison[['accuracy', 'precision_scam', 'recall_scam', 'f1_scam']].plot(
    kind='bar', figsize=(10, 5), ylim=(0, 1), color=['#4C78A8', '#F2CF5B', '#E45756', '#72B7B2']
)
plt.title('External-test performance: traditional versus transformer features')
plt.ylabel('Score')
plt.xlabel('')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 11: External confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for axis, (name, predicted) in zip(axes, [
    ('TF-IDF baseline', baseline_external_pred),
    ('MiniLM embeddings', transformer_external_pred),
]):
    matrix = confusion_matrix(external['label'], predicted, labels=[0, 1])
    ConfusionMatrixDisplay(matrix, display_labels=['Legitimate', 'Scam']).plot(
        ax=axis, cmap='Blues', colorbar=False
    )
    axis.set_title(name)
plt.tight_layout()
plt.show()

## Step 6 — Inspect agreements and disagreements
This shows whether MiniLM fixes baseline mistakes or simply makes different mistakes.

In [ ]:
# Cell 12: Add both model outputs to each external email
model_predictions = external.copy()
model_predictions['baseline_prediction'] = baseline_external_pred
model_predictions['baseline_scam_probability'] = baseline_external_prob
model_predictions['minilm_prediction'] = transformer_external_pred
model_predictions['minilm_scam_probability'] = transformer_external_prob
model_predictions['baseline_correct'] = model_predictions['baseline_prediction'] == model_predictions['label']
model_predictions['minilm_correct'] = model_predictions['minilm_prediction'] == model_predictions['label']

model_predictions['comparison_result'] = np.select(
    [
        model_predictions['baseline_correct'] & model_predictions['minilm_correct'],
        (~model_predictions['baseline_correct']) & model_predictions['minilm_correct'],
        model_predictions['baseline_correct'] & (~model_predictions['minilm_correct']),
        (~model_predictions['baseline_correct']) & (~model_predictions['minilm_correct']),
    ],
    ['both_correct', 'minilm_only_correct', 'baseline_only_correct', 'both_wrong'],
    default='unknown',
)
model_predictions['comparison_result'].value_counts().to_frame('emails')

In [ ]:
# Cell 13: Focus on real scams
scam_comparison = model_predictions.loc[model_predictions['label'] == 1].copy()
scam_summary = pd.DataFrame({
    'model': ['TF-IDF baseline', 'MiniLM embeddings'],
    'caught_scams': [int((scam_comparison['baseline_prediction'] == 1).sum()), int((scam_comparison['minilm_prediction'] == 1).sum())],
    'missed_scams': [int((scam_comparison['baseline_prediction'] == 0).sum()), int((scam_comparison['minilm_prediction'] == 0).sum())],
})
scam_summary

## Step 7 — Save the results

In [ ]:
# Cell 14: Export comparison tables
output_dir = Path('/content/hit401_part4_results')
output_dir.mkdir(exist_ok=True)
comparison.to_csv(output_dir / 'traditional_vs_minilm_metrics.csv', index=False)
scam_summary.to_csv(output_dir / 'caught_and_missed_by_model.csv', index=False)
model_predictions.to_csv(output_dir / 'model_prediction_comparison.csv', index=False)

archive_path = Path('/content/hit401_part4_results.zip')
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for file_path in output_dir.glob('*.csv'):
        archive.write(file_path, arcname=file_path.name)
print('Saved:', archive_path)

## What to tell the supervisor

> We compared a traditional TF-IDF representation with a modern pretrained MiniLM transformer representation. Both used Logistic Regression as the final classifier and the same training and testing emails. This isolates the effect of changing how the email language is represented. MiniLM produced contextual 384-dimensional embeddings, while TF-IDF represented word and phrase frequency. We evaluated both on familiar and source-separated external data and compared which scams each model caught or missed.

## Limitations
- MiniLM is pretrained and frozen; it was not fine-tuned on our email data.
- Long messages are truncated to the model's token limit.
- Better results would not prove that the transformer understands deception or user behaviour.
- Differences can still reflect dataset source, time period and label definitions.